## 10:00 AM — Planteando el cruce de segmentación
Jorge quiere saber: de los 15K clientes de la campaña, ¿cuántos son premium (>6 pedidos en 3 meses), regulares (2-6 pedidos) o nuevos (<2 pedidos)? Para responder vamos a usar tres datasets: campana_verano_clientes.csv (la lista de la campaña, con el cliente_id), pedidos.csv (el historial de pedidos del warehouse) y clientes.csv (la tabla maestra del CRM, de donde sacaremos el nombre y la fecha_registro, porque el archivo de campaña solo trae IDs y engagement). Esto es un par de LEFT JOIN con una agregación.

Piénsalo como un problema de la vida real: tienes dos listas. La primera es "personas invitadas a la fiesta" (clientes de la campaña). La segunda es "registro de compras del supermercado en los últimos 3 meses" (pedidos). Quieres saber, para cada invitado, cuántas veces ha comprado. Algunos invitados puede que NUNCA hayan comprado — y esos son los interesantes porque representan captación fallida.

La query se construye en 3 pasos lógicos: (1) contar pedidos por cliente en los últimos 3 meses, (2) cruzar esa cuenta con la lista de clientes de la campaña, (3) clasificar en segmentos según el número de pedidos. Vamos a usar CTEs (Common Table Expressions) para que cada paso sea legible:

In [7]:
import duckdb
import pandas as pd

ruta_absoluta = '/home/cristian_tobar/proyectos/Supermercado-online/data/campana_verano_clientes.csv'
ruta_absoluta_pedidos = '/home/cristian_tobar/proyectos/Supermercado-online/data/pedidos.csv'
ruta_abs_clientes = '/home/cristian_tobar/proyectos/Supermercado-online/data/clientes.csv'

# 1. Usar Pandas para leer el archivo (maneja los nulos sin problema)
df_campana = pd.read_csv(ruta_absoluta)
df_pedidos = pd.read_csv(ruta_absoluta_pedidos)
df_clientes = pd.read_csv(ruta_abs_clientes)

# 2. Conectar a DuckDB
con = duckdb.connect()

# 3. DuckDB puede leer directamente la variable 'df_ventas' de Pandas
ventas = con.execute("""
    SELECT * 
    FROM df_campana 
""").df()

display(ventas.head())

,cliente_id,campana,fecha_envio,canal_contacto,abrio_email,clickeo_link,compro,primer_compra_campana
0,104098,verano_2024,2024-06-22,email,True,False,False,None
1,108728,verano_2024,2024-06-22,email,True,False,False,None
2,113748,verano_2024,2024-06-22,email,False,<NA>,False,None
3,108519,verano_2024,2024-06-29,sms,<NA>,<NA>,False,None
4,101770,verano_2024,2024-06-29,sms,<NA>,<NA>,False,None


In [5]:
pedidos = con.execute("""
    SELECT * 
    FROM df_pedidos
""").df()
display(pedidos.head())

,pedido_id,cliente_id,tienda_id,fecha_pedido,estado,total_bruto,total_descuentos,total_neto,metodo_pago,codigo_postal_envio,hora_entrega
0,5000001,108333.0,1,2024-07-22 10:16:56,entregado,28.78,0.0,28.78,tarjeta,8019,2024-07-22 12:16:06
1,5000002,113685.0,2,2024-06-27 12:29:22,entregado,60.72,0.0,60.72,paypal,28046,2024-06-27 14:10:01
2,5000003,100603.0,4,2024-06-15 12:10:14,entregado,38.29,0.0,38.29,bizum,41015,2024-06-15 14:32:40
3,5000004,113389.0,1,2024-05-05 20:53:16,devuelto,13.27,0.0,13.27,paypal,8020,None
4,5000005,109525.0,5,2024-07-31 10:16:39,entregado,34.48,0.0,34.48,efectivo_entrega,48011,2024-07-31 12:46:23


In [8]:
clientes = con.execute("""
    SELECT * 
    FROM df_clientes
""").df()
display(clientes.head())

,cliente_id,nombre,email,telefono,fecha_registro,ciudad,codigo_postal,canal_captacion,segmento,fecha_ultima_compra
0,104098,Teodosio Ángel Adán,arandaangelino@example.org,+34126009103,2024-03-17,Madrid,28045,instagram_ads,regular,2024-08-27
1,108728,Heliodoro Marín Puerta,cerrolucho@example.net,+34861007532,2021-08-12,Barcelona,8007,google_ads,regular,2024-08-12
2,113748,Loida Canals-Alvarez,paz82@example.net,+34639401453,2024-07-25,Madrid,28003,tv,nuevo,2024-08-24
3,108519,Germán Garzón Torrent,hmanzano@example.com,+34933043058,2023-07-10,Barcelona,8006,google_ads,inactivo,2023-05-13
4,101770,Tristán Modesto Espejo Juliá,flor96@example.com,+34045398048,2021-03-23,Madrid,28001,organico,regular,2024-04-11


In [14]:
consulta = """
-- Paso 1: Contar pedidos por cliente
WITH pedidos_recientes AS (
    SELECT cliente_id, COUNT(*) AS pedidos_3m
    FROM df_pedidos
    WHERE fecha_pedido >= '2024-06-16'  
      AND estado NOT IN ('cancelado')   
    GROUP BY cliente_id
),

-- Paso 2: Cruzar con los clientes de la campaña y la tabla maestra
segmentacion AS (  -- <-- Aquí estaba el error (decía "segmentaicon")
    SELECT
        c.cliente_id,
        m.nombre,
        m.email,
        m.fecha_registro,
        COALESCE(p.pedidos_3m, 0) AS pedidos_ultimos_3m,
        CASE
            WHEN COALESCE(p.pedidos_3m, 0) > 6 THEN 'premium'
            WHEN COALESCE(p.pedidos_3m, 0) >= 2 THEN 'regular'
            ELSE 'nuevo'
        END AS segmento
    FROM df_campana c
    LEFT JOIN df_clientes m ON c.cliente_id = m.cliente_id
    LEFT JOIN pedidos_recientes p ON c.cliente_id = p.cliente_id
)

-- Paso 3: Resultado ordenado por actividad
SELECT * 
FROM segmentacion
ORDER BY pedidos_ultimos_3m DESC;
"""

# Ejecutar la consulta limpia directamente
consulta_final = con.execute(consulta).df()
display(consulta_final.head(5))

,cliente_id,nombre,email,fecha_registro,pedidos_ultimos_3m,segmento
0,111546,Lara Dolores Nieto Reig,cuevasblas@example.org,2024-01-22,23,premium
1,109130,Montserrat Benítez Sandoval,upoza@example.net,2024-07-13,21,premium
2,114200,Tomás Valenciano,maria-luisa88@example.com,2023-10-04,19,premium
3,113259,Concha Zabala Navas,calvoisaac@example.com,2020-07-18,19,premium
4,114632,Ani Olivares Echevarría,baudeliovergara@example.com,2021-03-09,19,premium


In [17]:
# Filtrar usando sintaxis SQL tradicional
clientes_cero_pedidos_sql = con.execute("""
    SELECT *
    FROM consulta_final
    WHERE pedidos_ultimos_3m = 0
""").df()

display(clientes_cero_pedidos_sql)

,cliente_id,nombre,email,fecha_registro,pedidos_ultimos_3m,segmento
0,104355,Rufina Barrena,leon95@example.org,2021-01-29,0,nuevo
1,100540,África Conde Cantero,fabianamontes@example.com,2020-12-17,0,nuevo
2,106621,Osvaldo Bolaños Pérez,jovececilio@example.com,2022-07-05,0,nuevo
3,100874,Daniel Castrillo Sosa,pbermudez@example.com,2023-01-19,0,nuevo
4,102301,Fabricio Barros Manzano,zcasals@example.net,2020-07-06,0,nuevo
...,...,...,...,...,...,...
6957,105006,Judith Valderrama Luís,wcalderon@example.org,2020-10-14,0,nuevo
6958,103477,Consuelo Lozano Olmedo,vmendizabal@example.org,2024-06-10,0,nuevo
6959,114998,José Ángel Hoz Peñas,macariaalcalde@example.com,2021-03-22,0,nuevo
6960,112174,Ana del Bayo,cantonregulo@example.org,2024-06-13,0,nuevo
